# EvoCA — scans notebook

Tests the C core (ctypes), verifies exact GoL, and launches the SDL2 interactive display.

Run cells in order.  **Build first**, then proceed.

## 1  Build

In [1]:
import subprocess, sys, os
root = os.path.abspath('')
# macOS: gcc -dynamiclib; Linux: gcc -shared
import platform
flag = '-dynamiclib' if platform.system() == 'Darwin' else '-shared'
ext  = 'dylib'      if platform.system() == 'Darwin' else 'so'
cmd  = ['gcc', '-O2', '-Wall', '-fPIC', flag,
        '-o', f'C/libevoca.{ext}', 'C/evoca.c']
r = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print(r.stdout or '(no stdout)')
if r.returncode != 0:
    print('STDERR:', r.stderr, file=sys.stderr)
    raise RuntimeError('Build failed')
print('Build OK')

(no stdout)
Build OK


## 2  Imports

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import time
from pprint import pprint

from python.evoca_py import EvoCA, make_gol_lut, LUT_BYTES, lut_bit_index
from python.evoca_py import unpack_lut, available_state_init
from python.display  import run as sdl_run
from python.controls import run_with_controls
from python.evoca_py import import_run
from python.controls import available_probes
from python.evoca_explore import evoca_from_scan, evoca_from_scan_top
from python.evoca_explore import evoca_from_scan_top, EVO_METRICS, SPATIAL_METRICS
from python.evoca_explore import nearest_params, nearest_evo, nearest_spatial


# Scans

I had Claude set up a script to scan parameter values, searching for 

- Reaction-diffusion complex spatial structure
- high evolution, using activity metrics

We completed 4 scans.  They are in subfolders of the Scans directory, in each subfolder there are notes on the scans, and a summary comparison in Scans/2026-04-27_scan_analysis.md.



# Scan imports

End-to-end works. From notebook you can now do:
```
from evoca_explore import evoca_from_scan, evoca_from_scan_top
from evoca_py import import_run
from controls import run_with_controls
```
### Option A: pick a specific config_idx from the CSV
```
path = evoca_from_scan('Scans/2026-04-27_initial', config_idx=49,
                     descriptor='best_corr_length',
                     probes={'activity': True, 'q_activity': True,
                             'n_activity': True, 'env_food': True,
                             'priv_food': True, 'ts': True},
                     colormode=4)
sim, kw = import_run(path)
run_with_controls(sim, **kw)
```
### Option B: dump top-K by composite score
```
results = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=5,
                            probes={...}, colormode=4)
```
**results is [(config_idx, path), ...]**
```
sim, kw = import_run(results[0][1])
run_with_controls(sim, **kw)
```

In [4]:
scan = pd.read_csv('Scans/2026-04-27_initial/results.csv')

In [5]:
scan.head()

,F_std_final,F_std_mean,F_std_temporal_std,N,alive_density_final,alive_density_mean,alive_density_temporal_std,config_idx,correlation_length_final,correlation_length_mean,...,param_food_inc,param_gdiff,param_m_scale,param_mu_egene,param_mu_lut,param_restricted_mu,param_tax,seed,shadow,unique_top_genomes
0,0.000000,0.015025,0.039278,256,0.000000,0.011225,0.069448,11,0.0,9.529412,...,0.015,0.010,0.6,0.001,0.030,True,0.035,0,True,16
1,0.000000,0.026440,0.068274,256,0.000000,0.013800,0.070772,4,0.0,8.039216,...,0.015,0.005,0.4,0.001,0.030,False,0.025,0,True,14
2,0.000000,0.029890,0.068056,256,0.000000,0.014282,0.071805,13,0.0,11.705882,...,0.015,0.005,0.6,0.001,0.010,False,0.025,0,True,25
3,0.451295,0.437447,0.065362,256,0.495056,0.467331,0.090361,0,16.0,25.980392,...,0.015,0.020,0.6,0.001,0.003,True,0.025,0,True,47
4,0.017998,0.033617,0.073552,256,0.999969,0.980325,0.084667,8,1.0,8.333333,...,0.025,0.020,0.4,0.001,0.030,False,0.025,0,True,1


In [6]:
scan.columns

Index(['F_std_final', 'F_std_mean', 'F_std_temporal_std', 'N',
       'alive_density_final', 'alive_density_mean',
       'alive_density_temporal_std', 'config_idx', 'correlation_length_final',
       'correlation_length_mean', 'correlation_length_temporal_std', 'error',
       'excess_activity_final', 'excess_activity_slope', 'extinct',
       'final_pop', 'init', 'largest_patch_final', 'largest_patch_mean',
       'largest_patch_temporal_std', 'min_pop', 'n_distinct_genomes_final',
       'n_distinct_genomes_mean', 'n_distinct_genomes_temporal_std',
       'n_patches_final', 'n_patches_mean', 'n_patches_temporal_std',
       'n_steps', 'param_food_inc', 'param_gdiff', 'param_m_scale',
       'param_mu_egene', 'param_mu_lut', 'param_restricted_mu', 'param_tax',
       'seed', 'shadow', 'unique_top_genomes'],
      dtype='str')

In [9]:
path = evoca_from_scan('Scans/2026-04-27_initial', config_idx=49,
                     descriptor='best_corr_length',
                     probes={'activity': True, 'q_activity': True,
                             #'n_activity': True, 'env_food': True,
                             'priv_food': True, 'ts': True},
                     colormode=4)
sim, kw = import_run(path)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6359527424)>

In [3]:
results = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=10,
                            probes={'activity': True,
                                   'ts':True}, colormode=4)

In [4]:
results

[(49, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_1_cfg49_score0.75.evoca'),
 (30, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_2_cfg30_score0.72.evoca'),
 (70, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_3_cfg70_score0.68.evoca'),
 (34, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_4_cfg34_score0.63.evoca'),
 (35, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_5_cfg35_score0.62.evoca'),
 (3, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_6_cfg3_score0.61.evoca'),
 (87, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_7_cfg87_score0.60.evoca'),
 (111, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_8_cfg111_score0.58.evoca'),
 (82, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_9_cfg82_score0.55.evoca'),
 (39, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_10_cfg39_score0.55.evoca')]

In [6]:
sim, kw = import_run(results[8][1],N=512)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6320009216)>

## nearest to an interesting entry


**Same return shape as evoca_from_scan_top: [(config_idx, path), ...]**
```
  r = nearest_params(scan_dir, target_config_idx=34, n=5, probes={...})
  r = nearest_evo(scan_dir, 34, n=5, probes={...})
  r = nearest_spatial(scan_dir, 34, n=5, probes={...})

  sim, kw = import_run(r[0][1], N=512)
  run_with_controls(sim, **kw)
```

  Each prints the per-rank distances during the call so you can see at a glance how close the neighbours actually are.
   Distance metrics: Euclidean in [0,1]-normalised param space for nearest_params; mean rank-difference for
  nearest_evo and nearest_spatial (smaller = more similar).

In [5]:
results[6]

(87, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_7_cfg87_score0.60.evoca')

In [6]:
r = nearest_params(target_config_idx=87, n=5, probes={})

writing recipes to /tmp/evoca_scan_neighbors/near_param_zl9rm8jp
  rank 1: cfg139   d= 0.670
  rank 2: cfg85    d= 0.781
  rank 3: cfg63    d= 0.892
  rank 4: cfg34    d= 0.931
  rank 5: cfg79    d= 0.975


In [7]:
r

[(139,
  '/tmp/evoca_scan_neighbors/near_param_zl9rm8jp/2026-04-27_near_param_1_cfg139_d0.67.evoca'),
 (85,
  '/tmp/evoca_scan_neighbors/near_param_zl9rm8jp/2026-04-27_near_param_2_cfg85_d0.78.evoca'),
 (63,
  '/tmp/evoca_scan_neighbors/near_param_zl9rm8jp/2026-04-27_near_param_3_cfg63_d0.89.evoca'),
 (34,
  '/tmp/evoca_scan_neighbors/near_param_zl9rm8jp/2026-04-27_near_param_4_cfg34_d0.93.evoca'),
 (79,
  '/tmp/evoca_scan_neighbors/near_param_zl9rm8jp/2026-04-27_near_param_5_cfg79_d0.97.evoca')]

In [14]:
sim, kw = import_run(r[4][1],N=512)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6388330496)>


## Top evo, top spatial

Use evoca_from_scan_top with score_keys set to the metric set you want:
```
from python.evoca_explore import evoca_from_scan_top, EVO_METRICS, SPATIAL_METRICS
```
**Top-K by evolution alone**
```
r_evo = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=5,
                          score_keys=EVO_METRICS,
                          descriptor_prefix='top_evo',
                          probes={...}, colormode=4)

```
**Top-K by spatial alone**
```
r_spatial = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=5,
                              score_keys=SPATIAL_METRICS,
                              descriptor_prefix='top_spatial',
                              probes={...}, colormode=4)

sim, kw = import_run(r_evo[0][1], N=512)
run_with_controls(sim, **kw)
```
The composite score is the same mean(normalised(metric)) formula, just over a single axis of metrics. EVO_METRICS
and SPATIAL_METRICS are the same module-level lists used as defaults by nearest_evo / nearest_spatial, so passing
one of them flips evoca_from_scan_top from "balanced ranker" to "pure-evo" or "pure-spatial".

Setting descriptor_prefix='top_evo' (or 'top_spatial') avoids filename collisions with the default top_* files from
a balanced run.


In [3]:
r_evo = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=5,
                          score_keys=EVO_METRICS,
                          descriptor_prefix='top_evo',
                          probes={'activity': True,
                                   'ts':True}, colormode=4)


In [4]:
r_evo

[(30,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_1_cfg30_score0.90.evoca'),
 (49,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_2_cfg49_score0.89.evoca'),
 (77,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_3_cfg77_score0.81.evoca'),
 (35,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_4_cfg35_score0.81.evoca'),
 (39,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_5_cfg39_score0.79.evoca')]

In [8]:
sim, kw = import_run(r_evo[2][1],N=512)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6410006528)>

EvoCA SDL: starting  N=512 px=2  probes=['--ts=psm_6950c4cd']


In [6]:
import numpy as np, ctypes
N = sim.N

# How many distinct colours are alive cells actually showing?
pixels = np.zeros(N*N, dtype=np.int32)
sim.colorize(pixels, 0)
alive = np.ctypeslib.as_array(sim._lib.evoca_get_alive(), shape=(N*N,))
distinct = len(np.unique(pixels[alive == 1]))
print(f"distinct colours among alive cells: {distinct}")

distinct colours among alive cells: 2


In [7]:
# Verify we really are at t=0
print(f"step = {sim.get_step()}")

pixels = np.zeros(N*N, dtype=np.int32)
sim.colorize(pixels, 0)
alive  = np.ctypeslib.as_array(sim._lib.evoca_get_alive(), shape=(N*N,))
v_curr = np.ctypeslib.as_array(sim._lib.evoca_get_v(),     shape=(N*N,))

m_v1 = (alive == 1) & (v_curr == 1)
m_v0 = (alive == 1) & (v_curr == 0)
if m_v1.any():
  print(f"alive v=1 colour: {hex(int(pixels[m_v1][0]) & 0xFFFFFFFF)} "
        f"(expect 0xffffffff white)")
if m_v0.any():
  print(f"alive v=0 colour: {hex(int(pixels[m_v0][0]) & 0xFFFFFFFF)} "
        f"(expect 0xff333333 dark grey)")

# Also: verify cell 0 has GoL LUT bytes
from evoca_py import make_gol_lut
import_lut = np.ctypeslib.as_array(sim._lib.evoca_get_lut(),
                                  shape=(N*N*32,)).reshape(N*N, 32)
first_alive = int(np.argmax(alive))
gol = np.array(make_gol_lut())
print(f"first alive cell #{first_alive}: GoL match = "
    f"{np.array_equal(import_lut[first_alive], gol)}")

step = 0
alive v=1 colour: 0xffffffff (expect 0xffffffff white)
alive v=0 colour: 0xff333333 (expect 0xff333333 dark grey)
first alive cell #0: GoL match = True


# Scan 3

In [3]:
dat = pd.read_csv('Scans/2026-04-27_push_edges/results.csv')

In [4]:
len(dat)

250

In [5]:
dat.columns

Index(['F_std_final', 'F_std_mean', 'F_std_temporal_std', 'N',
       'alive_density_final', 'alive_density_mean',
       'alive_density_temporal_std', 'config_idx', 'correlation_length_final',
       'correlation_length_mean', 'correlation_length_temporal_std', 'error',
       'excess_activity_final', 'excess_activity_slope', 'extinct',
       'final_pop', 'init', 'largest_patch_final', 'largest_patch_mean',
       'largest_patch_temporal_std', 'min_pop', 'n_distinct_genomes_final',
       'n_distinct_genomes_mean', 'n_distinct_genomes_temporal_std',
       'n_patches_final', 'n_patches_mean', 'n_patches_temporal_std',
       'n_steps', 'param_food_inc', 'param_gdiff', 'param_m_scale',
       'param_mu_egene', 'param_mu_lut', 'param_restricted_mu', 'param_tax',
       'seed', 'shadow', 'unique_top_genomes'],
      dtype='str')

In [4]:
#scan 3

results = evoca_from_scan_top('Scans/2026-04-27_push_edges', top_k=20,
                            probes={'activity': True,
                                   'ts':True}, colormode=4)

In [4]:
[(i,results[i]) for i in range(len(results))]

[(0,
  (236,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_1_cfg236_score0.90.evoca')),
 (1,
  (246,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_2_cfg246_score0.88.evoca')),
 (2,
  (30, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_3_cfg30_score0.85.evoca')),
 (3,
  (247,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_4_cfg247_score0.82.evoca')),
 (4,
  (178,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_5_cfg178_score0.80.evoca')),
 (5,
  (36, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_6_cfg36_score0.80.evoca')),
 (6,
  (107,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_7_cfg107_score0.80.evoca')),
 (7,
  (211,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_8_cfg211_score0.79.evoca')),
 (8,
  (98, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_9_cfg98_score0.78.evoca')),
 (9,
  (221,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_10_cfg221_score0.77.evoca')),
 (10,
  (115,
   '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_11_cfg115_score0.76.evoca')),
 (11,
  (184,
   '/Use

In [8]:
dat['excess_activity_slope'][1]

np.float64(117.69806655829764)

In [47]:
[(j,i,float(dat['excess_activity_slope'][i])) for j,i in enumerate([x[0] for x in results])]

[(0, 236, 182.48033237771543),
 (1, 246, 150.92392498891678),
 (2, 30, 144.07116222846167),
 (3, 247, 424.1235655977539),
 (4, 178, 0.9209599527116888),
 (5, 36, 1.7498649918723226),
 (6, 107, 23.132225387911923),
 (7, 211, -31.828439131077275),
 (8, 98, 259.83800797990256),
 (9, 221, 20.762342544702236),
 (10, 115, 85.40022012708737),
 (11, 184, 123.9285881483671),
 (12, 201, -9.18298596128269),
 (13, 43, 25.957762317127255),
 (14, 196, 182.66774523422495),
 (15, 102, 3.858171656568641),
 (16, 106, 1.3026465494310635),
 (17, 202, 11.287842175262313),
 (18, 148, 42.12504725875574),
 (19, 64, 39.98139725136696)]

In [48]:
[(j,i,float(dat['n_distinct_genomes_temporal_std'][i])) for j,i in enumerate([x[0] for x in results])]

[(0, 236, 2282.3184157252563),
 (1, 246, 2230.099870996354),
 (2, 30, 1977.1107998599616),
 (3, 247, 3058.018028951018),
 (4, 178, 3152.4232880853247),
 (5, 36, 12820.553222087306),
 (6, 107, 1167.2865895227494),
 (7, 211, 943.3466469513992),
 (8, 98, 2453.521800658449),
 (9, 221, 2999.708421570376),
 (10, 115, 2859.805083500614),
 (11, 184, 2792.1689681667544),
 (12, 201, 626.787798922093),
 (13, 43, 1803.3888542188577),
 (14, 196, 2019.5363087485955),
 (15, 102, 2530.9851189966253),
 (16, 106, 11264.102549670735),
 (17, 202, 7387.284233396699),
 (18, 148, 3489.5890917565357),
 (19, 64, 1473.456330648372)]

**look through the top results**

Vary n to get simulation.

Add/change probes, whatever, to kw before calling `run_with_controls(sim, **kw)`

In [14]:
n=8
sim, kw = import_run(results[n][1],N=512)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6285242368)>

In [15]:
kw

{'colormode': 4, 'probes': {'activity': True, 'ts': True}}

In [8]:
results[8]

(98, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_9_cfg98_score0.78.evoca')

In [5]:
revo = nearest_evo(98,10)

writing recipes to /tmp/evoca_scan_neighbors/near_evo_ydlcspd6
  rank 1: cfg236   d=10.250
  rank 2: cfg144   d=12.000
  rank 3: cfg164   d=12.250
  rank 4: cfg115   d=12.500
  rank 5: cfg184   d=14.000
  rank 6: cfg246   d=18.250
  rank 7: cfg186   d=18.750
  rank 8: cfg201   d=19.500
  rank 9: cfg49    d=19.750
  rank 10: cfg2     d=20.500


In [6]:
revo[0]


(236,
 '/tmp/evoca_scan_neighbors/near_evo_ydlcspd6/2026-04-27_near_evo_1_cfg236_d10.25.evoca')

In [27]:
sim, kw = import_run(revo[9][1],N=512)
kw['probes'] = {'n_activity': True, 'ts': True}
kw['colormode'] = 1
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6344830976)>

EvoCA SDL: starting  N=512 px=2  probes=['--ts=psm_efb81b4a']
EvoCA SDL: probe SharedMemory open failed: [Errno 2] No such file or directory: '/--n-activity=psm_54fbfe85'
EvoCA SDL: n_activity shm opened (256x512)
EvoCA SDL: ts shm opened (6 traces)


In [26]:
sim.params()

{'N': 512,
 'food_inc': 0.01,
 'm_scale': 1.2,
 'gdiff': 0.12,
 'mu_lut': 0.01,
 'mu_egene': 0.01,
 'tax': 0.04,
 'restricted_mu': 0}

In [12]:
[x for x in dir(sim) if 'par' in x]

['_init_metaparams', '_state_params', 'params', 'params_str', 'state_params']

In [15]:
[x for x in dir(sim) if 'init' in x]

['__init__', '__init_subclass__', '_init_metaparams', 'init']

In [14]:
sim.state_params

{'lut': 'gol',
 'egenome': 'uniform',
 'egenome_value': 3,
 'v_density': 0.5,
 'f_init': 0.5,
 'F': 'uniform',
 'F_init': 1.0,
 'alive': 'halfplane',
 'alive_axis': 0}

In [16]:
EVO_METRICS

['n_distinct_genomes_mean',
 'n_distinct_genomes_temporal_std',
 'unique_top_genomes',
 'excess_activity_slope']

In [17]:

dat[EVO_METRICS].head()

,n_distinct_genomes_mean,n_distinct_genomes_temporal_std,unique_top_genomes,excess_activity_slope
0,13035.293532,2687.012558,107,53.095207
1,7735.353234,1487.230710,49,117.698067
2,14094.985075,8224.878093,188,23.333574
3,16215.835821,4290.050721,164,29.521504
4,5257.427861,2753.216161,162,45.598798


## Top evo from scan 3

In [30]:
r_evo = evoca_from_scan_top('Scans/2026-04-27_push_edges', top_k=20,
                          score_keys=EVO_METRICS,
                          descriptor_prefix='top_evo',
                          probes={'n_activity': True, 'ts': True}, colormode=1)

In [32]:
r_evo

[(30,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_1_cfg30_score0.64.evoca'),
 (221,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_2_cfg221_score0.59.evoca'),
 (211,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_3_cfg211_score0.59.evoca'),
 (36,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_4_cfg36_score0.58.evoca'),
 (107,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_5_cfg107_score0.58.evoca'),
 (64,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_6_cfg64_score0.58.evoca'),
 (247,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_7_cfg247_score0.58.evoca'),
 (202,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_8_cfg202_score0.58.evoca'),
 (178,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_9_cfg178_score0.57.evoca'),
 (246,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_10_cfg246_score0.56.evoca'),
 (236,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_11_cfg236_score0.56.evoca'),
 (137,
  '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_evo_

In [36]:
sim, kw = import_run(r_evo[4][1],N=512)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6344830976)>

### playing with initial conditions

In [45]:
sim, kw = import_run(r_evo[6][1],N=512)
params_state = dict(lut='gol', lut_n_init=1,
                  alive='fraction',
                  alive_fraction=0.1,
                  egenome='uniform',
                    egenome_value=0b000011,
                  f_init=0.1,
                  F_init=0.5)
sim.state(**params_state)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6344830976)>

EvoCA SDL: starting  N=512 px=2  probes=['--ts=psm_d79317e8']
EvoCA SDL: probe SharedMemory open failed: [Errno 2] No such file or directory: '/--n-activity=psm_6d06cb94'
EvoCA SDL: n_activity shm opened (256x512)
EvoCA SDL: ts shm opened (6 traces)
